# 🏥 Disease Prediction from Medical Data
## CodeAlpha Machine Learning Internship — Task 4
**Author:** Rose Sharma  
**Objective:** Predict possibility of Heart Disease using patient medical data  
**Algorithms:** SVM, Logistic Regression, Random Forest, XGBoost  

---

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False
    print('⚠️ XGBoost not installed — skipping XGBoost')

import pickle, os
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries imported!')

## 📂 Step 2 — Load Heart Disease Dataset

In [ ]:
# Load UCI Heart Disease dataset
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/heart.csv'
try:
    df = pd.read_csv(url)
    print('✅ Dataset loaded from URL!')
except:
    # Fallback: generate realistic heart disease data
    print('⚠️ URL failed — generating synthetic heart disease data...')
    np.random.seed(42)
    n = 1025
    df = pd.DataFrame({
        'age':      np.random.randint(29, 77, n),
        'sex':      np.random.randint(0, 2, n),
        'cp':       np.random.randint(0, 4, n),
        'trestbps': np.random.randint(94, 200, n),
        'chol':     np.random.randint(126, 564, n),
        'fbs':      np.random.randint(0, 2, n),
        'restecg':  np.random.randint(0, 3, n),
        'thalach':  np.random.randint(71, 202, n),
        'exang':    np.random.randint(0, 2, n),
        'oldpeak':  np.round(np.random.uniform(0, 6.2, n), 1),
        'slope':    np.random.randint(0, 3, n),
        'ca':       np.random.randint(0, 5, n),
        'thal':     np.random.randint(0, 4, n),
    })
    score = (
        -df['age']*0.02 + df['cp']*0.3 - df['trestbps']*0.01
        - df['chol']*0.001 + df['thalach']*0.01 - df['exang']*0.3
        - df['oldpeak']*0.2 + np.random.normal(0,0.3,n)
    )
    df['target'] = (score > score.mean()).astype(int)

print(f'   Shape  : {df.shape}')
print(f'   Target : 1=Heart Disease, 0=No Disease')
print(f'   Disease: {df["target"].sum()} ({df["target"].mean()*100:.1f}%)')
df.head()

In [ ]:
# Feature descriptions
feature_info = {
    'age':      'Age of patient',
    'sex':      'Sex (1=Male, 0=Female)',
    'cp':       'Chest pain type (0-3)',
    'trestbps': 'Resting blood pressure (mmHg)',
    'chol':     'Serum cholesterol (mg/dl)',
    'fbs':      'Fasting blood sugar > 120 mg/dl',
    'restecg':  'Resting ECG results (0-2)',
    'thalach':  'Maximum heart rate achieved',
    'exang':    'Exercise induced angina',
    'oldpeak':  'ST depression induced by exercise',
    'slope':    'Slope of peak exercise ST segment',
    'ca':       'Number of major vessels (0-4)',
    'thal':     'Thalassemia (0=Normal, 1=Fixed, 2=Reversible)',
    'target':   '🎯 Heart Disease (1=Yes, 0=No)'
}
print('📖 Feature Descriptions:')
for f, d in feature_info.items():
    print(f'   {f:10s}: {d}')

## 🔍 Step 3 — EDA

In [ ]:
print(df.describe().round(2))
print(f'\nMissing values: {df.isnull().sum().sum()}')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Heart Disease Target Distribution', fontsize=14, fontweight='bold')

labels = ['No Disease', 'Heart Disease']
counts = df['target'].value_counts().sort_index()
colors_pie = ['#27ae60','#e74c3c']

axes[0].bar(labels, counts.values, color=colors_pie, alpha=0.85, edgecolor='white', width=0.5)
axes[0].set_title('Count')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+5, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, colors=colors_pie,
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proportion')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Key feature analysis
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Key Features vs Heart Disease', fontsize=14, fontweight='bold')
axes = axes.flatten()
key_features = ['age', 'thalach', 'chol', 'trestbps', 'oldpeak', 'cp']

for i, feat in enumerate(key_features):
    for label, color in zip([0,1], ['#27ae60','#e74c3c']):
        axes[i].hist(df[df['target']==label][feat], bins=20,
                     alpha=0.6, color=color,
                     label=['No Disease','Heart Disease'][label])
    axes[i].set_title(feat, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.savefig('feature_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, square=True)
plt.title('Correlation Heatmap — Heart Disease Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## ✂️ Step 4 — Preprocessing & Split

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'✅ Split done: Train={X_train.shape} | Test={X_test.shape}')

## 🤖 Step 5 — Train All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'SVM':                 SVC(probability=True, random_state=42, kernel='rbf'),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42)
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(random_state=42, eval_metric='logloss')

results = {}
for name, m in models.items():
    m.fit(X_train_s, y_train)
    y_pred = m.predict(X_test_s)
    y_prob = m.predict_proba(X_test_s)[:,1]
    results[name] = {
        'model':     m,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_prob)
    }
    print(f'✅ {name:<22} Acc:{results[name]["accuracy"]:.3f} F1:{results[name]["f1"]:.3f} AUC:{results[name]["roc_auc"]:.3f}')

In [ ]:
# ROC curves + metrics comparison
colors_list = ['#1a3c6e','#e74c3c','#27ae60','#f39c12']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Disease Prediction — Model Comparison', fontsize=14, fontweight='bold')

for (name, res), color in zip(results.items(), colors_list):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f'{name} (AUC={res["roc_auc"]:.3f})')

axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(loc='lower right')

metrics_names = ['accuracy','precision','recall','f1','roc_auc']
x = np.arange(len(metrics_names))
w = 0.8 / len(results)
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics_names]
    axes[1].bar(x + i*w - 0.4, vals, w, label=name,
                color=colors_list[i], alpha=0.85)

axes[1].set_xticks(x)
axes[1].set_xticklabels(['Accuracy','Precision','Recall','F1','ROC-AUC'])
axes[1].set_title('All Metrics Comparison')
axes[1].legend()
axes[1].set_ylim(0,1.1)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Best model confusion matrix
best_name = max(results, key=lambda x: results[x]['roc_auc'])
best = results[best_name]
print(f'🏆 Best Model: {best_name}')

cm = confusion_matrix(y_test, best['y_pred'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Disease','Heart Disease'],
            yticklabels=['No Disease','Heart Disease'])
plt.title(f'{best_name} — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print(classification_report(y_test, best['y_pred'],
      target_names=['No Disease','Heart Disease']))

In [ ]:
# Feature importance (Random Forest)
rf = results['Random Forest']['model']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)\
             .sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.plot(kind='bar', color='#e74c3c', alpha=0.85, edgecolor='white')
plt.title('Random Forest — Feature Importance for Disease Prediction',
          fontsize=12, fontweight='bold')
plt.xlabel('Medical Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Save best model
os.makedirs('model', exist_ok=True)
with open('model/disease_model.pkl','wb') as f:
    pickle.dump(results[best_name]['model'], f)
with open('model/scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

print(f'✅ Best model ({best_name}) saved!')
print(f'\n📊 FINAL SUMMARY')
print('='*60)
for name, res in results.items():
    print(f'  {name:<22} Acc:{res["accuracy"]:.3f} F1:{res["f1"]:.3f} AUC:{res["roc_auc"]:.3f}')
print('='*60)
print(f'  🏆 Best: {best_name}')
print(f'  Author: Rose Sharma | CodeAlpha ML Internship — Task 4')